# ViT-LENS: Notebook 03 — TopK Sparse Autoencoder (SAE) Training
**Research Goal**: Train custom TopK SAE ($d_{in}=768, d_{sae}=3072, k=32$) on extracted ViT residual-stream activations with decoder column normalization and resumable checkpointing.


In [ ]:
# Step 1: Initialize TopK SAE Architecture
import torch
from torch.utils.data import DataLoader
from src.utils import get_device, load_config
from src.topk_sae import TopKSparseAutoencoder
from src.sae_trainer import SAETrainer
from src.activation_extractor import ChunkedActivationDataset

device = get_device('auto')
config = load_config('config.yaml')

sae = TopKSparseAutoencoder(
    d_in=config['sae']['d_in'],
    d_sae=config['sae']['d_sae'],
    top_k=config['sae']['top_k']
).to(device)

print(f'Initialized TopK SAE: {sae.d_in} -> {sae.d_sae} (Expansion {sae.d_sae/sae.d_in:.1f}x) | k = {sae.top_k}')


In [ ]:
# Step 2: Initialize Chunked Activation DataLoader
act_dataset = ChunkedActivationDataset(config['extraction']['output_dir'])
print(f'Total patch activation vectors in dataset: {len(act_dataset):,}')

act_loader = DataLoader(
    act_dataset,
    batch_size=config['sae']['batch_size'],
    shuffle=True,
    num_workers=0
)


In [ ]:
# Step 3: Run SAE Training Loop with Checkpointing
trainer = SAETrainer(
    sae=sae,
    lr=config['sae']['lr'],
    weight_decay=config['sae']['weight_decay'],
    device=device,
    checkpoint_dir=config['sae']['checkpoint_dir'],
    dead_feature_threshold_steps=config['sae']['dead_feature_threshold_steps'],
    resample_dead_features=config['sae']['resample_dead_features']
)

history = trainer.fit(
    dataloader=act_loader,
    num_epochs=config['sae']['num_epochs'],
    save_every_epochs=config['sae']['save_every_epochs']
)
print('Training complete! Final checkpoint saved to checkpoints/sae_checkpoint_latest.pt')
